# 🚀 Notebook 03: Treinamento do Qwen 2.5 Coder 32B em BF16 Nativo
**Ambiente:** 1x GPU 96 GB VRAM + 180 GB RAM
**Modelo Alvo:** `Qwen/Qwen2.5-Coder-32B-Instruct`

Com **96 GB de VRAM**, desbloqueamos a especificação máxima do Plano Diretor:
- **Precisão:** Bfloat16 (BF16) nativo (~64 GB de pesos na VRAM) sem ruído de quantização.
- **Adaptadores LoRA:** $r=64, \alpha=128$ injetados em todas as camadas lineares de atenção e MLPs (`q_proj`, `k_proj`, `v_proj`, `o_proj`, `gate_proj`, `up_proj`, `down_proj`).
- **FlashAttention-2:** Janela de contexto estendida de **16.384 tokens** para auditoria de repositórios inteiros.
- **Otimizador:** Paged AdamW + Gradient Checkpointing nos 32 GB de VRAM restantes.
- **Persistência:** Checkpoints automáticos salvos no Google Drive a cada 100/250 passos com retomada automática (`resume_from_checkpoint=True`).


In [ ]:
# 1. Montagem do Drive e Checagem da GPU de 96 GB
from google.colab import drive
import os, torch, psutil
drive.mount('/content/drive')

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f'[*] GPU ATIVA: {gpu_name} ({vram_gb:.1f} GB VRAM)')
print(f'[*] RAM DISPONÍVEL: {psutil.virtual_memory().total / (1024**3):.1f} GB')

DRIVE_BASE = '/content/drive/MyDrive/cyber_llm_project'
TRAIN_DATA = os.path.join(DRIVE_BASE, 'training_ready/train.parquet')
CKPT_DIR = os.path.join(DRIVE_BASE, 'model_checkpoints')
FINAL_MODEL = os.path.join(CKPT_DIR, 'qwen32b_cyber_lora_final')
os.makedirs(CKPT_DIR, exist_ok=True)


## 2. Instalação do Unsloth e FlashAttention-2


In [ ]:
# Instalação otimizada para GPUs modernas com suporte a FlashAttention-2 e BF16 nativo
!pip install --no-deps -q 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'
!pip install --no-deps -q 'xformers<0.0.29' trl peft accelerate bitsandbytes datasets flash-attn


## 3. Carregamento do Qwen 2.5 Coder 32B em Bfloat16 Nativo com Contexto de 16k

> **Nota sobre a VRAM:** O modelo de 32B em BF16 ocupa ~64 GB de VRAM. Os ~32 GB restantes são gerenciados pelo Paged AdamW e Gradient Checkpointing.


In [ ]:
from unsloth import FastLanguageModel

# Configuração de Elite: 32B Base em BF16
MODEL_NAME = 'Qwen/Qwen2.5-Coder-32B-Instruct'
MAX_SEQ_LENGTH = 16384 # 16k tokens completos!

print(f'[*] Carregando {MODEL_NAME} em Bfloat16 nativo...')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False, # Treinamento em precisão BF16 pura sem ruído de quantização!
    dtype=torch.bfloat16,
)

# Injeta adaptadores LoRA de alta capacidade (r=64, alpha=128) exatamente como no Plano Diretor
model = FastLanguageModel.get_peft_model(
    model,
    r=64,
    lora_alpha=128,
    lora_dropout=0,
    bias='none',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    use_gradient_checkpointing='unsloth',
    random_state=42,
)

print('✓ Modelo 32B em BF16 carregado e adaptadores LoRA (r=64, a=128) injetados!')


## 4. Carregamento do Dataset Formatado com ChatML


In [ ]:
import pandas as pd
from datasets import Dataset

print(f'[*] Lendo dataset de treino: {TRAIN_DATA}')
df_train = pd.read_parquet(TRAIN_DATA)

def apply_chat_template(examples):
    texts = []
    for msgs in examples['messages']:
        formatted = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
        texts.append(formatted)
    return {'text': texts}

hf_dataset = Dataset.from_pandas(df_train)
hf_dataset = hf_dataset.map(apply_chat_template, batched=True, desc='Tokenização ChatML Qwen')
print(f'✓ Amostras prontas para o treinamento: {len(hf_dataset):,}')


## 5. Execução do Treinamento com Checkpoints Automáticos no Google Drive


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

# Verifica existência de checkpoints prévios para retomar automaticamente
resume_ckpt = None
if os.path.exists(CKPT_DIR):
    saved_ckpts = [os.path.join(CKPT_DIR, d) for d in os.listdir(CKPT_DIR) if d.startswith('checkpoint-')]
    if saved_ckpts:
        saved_ckpts.sort(key=os.path.getmtime)
        resume_ckpt = saved_ckpts[-1]
        print(f'[*] Checkpoint prévio encontrado! Retomando de: {resume_ckpt}')

training_args = TrainingArguments(
    output_dir=CKPT_DIR,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8, # Batch global efetivo = 16 amostras
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,
    num_train_epochs=1,
    max_steps=2000, # Ajuste a duração desejada
    logging_steps=10,
    save_steps=100, # Grava no Drive a cada 100 passos
    save_total_limit=3,
    bf16=True,
    optim='paged_adamw_8bit', # Gerencia perfeitamente a VRAM de 96 GB
    weight_decay=0.01,
    seed=42,
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=hf_dataset,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=4,
    packing=False,
    args=training_args,
)

print('===========================================================')
print('🚀 INICIANDO TREINAMENTO DO QWEN 2.5 CODER 32B EM BF16 NATIVO...')
print('===========================================================')
trainer_stats = trainer.train(resume_from_checkpoint=resume_ckpt)


## 6. Exportação dos Adaptadores LoRA para o Google Drive


In [ ]:
print(f'[*] Salvando adaptadores finais em: {FINAL_MODEL}')
model.save_pretrained(FINAL_MODEL)
tokenizer.save_pretrained(FINAL_MODEL)
print('✓ ADAPTADORES LORA SALVOS COM SUCESSO NO GOOGLE DRIVE!')
print('-> Prossiga para o Notebook 04 para Fusão de Pesos, Avaliação e Servidor vLLM!')
